# Completed section

In [3]:
import math
import torch
import torch.nn as nn
import numpy as np
from torch.nn import functional as F
from einops import rearrange

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
print(device)

class Patchify2D(nn.Module):
    def __init__(self, d_model: int, patch_size: int, color_channels: int):
        """
        input: (B, C, H, W)
        output: (B, N, D)
        """
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels=color_channels, out_channels=d_model, kernel_size=patch_size, stride=patch_size
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # LINEAR CxHxW -> (H/p * W/p)xCPP -> (NxCPP) @ (CPP x d) -> NxD
        # CONV   CxHxW -> DxH/pxW/p       -> DxN -> NxD
        return rearrange(self.conv(x), "... d h w -> ...  (h w) d")
        

class Depatchify2D(nn.Module):
    def __init__(self, d_model: int, patch_size: int, color_channels: int, height: int, width: int):
        """
        input: (B, N, D)
        output: (B, C, H, W)
        """
        super().__init__()
        self.height = height
        self.width = width
        self.patch_size = patch_size
        
        self.deconv = nn.ConvTranspose2d(in_channels=d_model, out_channels=color_channels, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h1 = self.height // self.patch_size
        w1 = self.width // self.patch_size
        # x # B, N, D -> B, H/p, W/p, D -> B, D, H/p, W/p
        # N = h // p * w // p
        x = rearrange(x, "... (h1 w1) d ->... d h1 w1", h1 = h1, w1 = w1)
        return self.deconv(x)
    
class FourierTimeEmbedding(nn.Module):
    def __init__(self, t_channels: int, w_min: float = 1.0, w_max: float = 10000.0):
        """
        input: (B)
        output: (B, T)
        """
        super().__init__()
        self.t_channels = t_channels
        half_dim = t_channels // 2
        
        w = w_min * ((w_max / w_min) ** ((torch.linspace(start=1, end=half_dim, steps=half_dim) - 1) / (half_dim - 1)))
        self.register_buffer("w", w) # non trainable

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        args = 2 * math.pi * torch.einsum('t, w -> tw', t, self.w)
        emb = math.sqrt(2.0 / self.t_channels) * torch.cat([torch.cos(args), torch.sin(args)], dim=1) # Batch x t_channels
        
        if self.t_channels % 2 == 1:  # zero pad
            emb = nn.functional.pad(emb, (0, 1), mode='constant')
        return emb
    
class MLP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        
        self.layers = nn.Sequential(
            nn.Linear(in_channels, out_channels),
            nn.GELU(),      
            nn.Linear(out_channels, out_channels)
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

cuda


## In prog

In [2]:
class RoPE(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return NotImplementedError

class EmbedND(nn.Module):
    def __init__(self, dim: int, theta: int, axes_dim: list[int]):
        super().__init__()
        self.dim = dim
        self.theta = theta
        self.axes_dim = axes_dim

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        n_axes = ids.shape[-1]
        emb = torch.cat(
            [rope(ids[..., i], self.axes_dim[i], self.theta) for i in range(n_axes)],
            dim=-3,
        )

        return emb.unsqueeze(1)


def rope(pos: torch.Tensor, dim: int, theta: int) -> torch.Tensor:
    assert dim % 2 == 0
    scale = torch.arange(0, dim, 2, dtype=pos.dtype, device=pos.device) / dim
    omega = 1.0 / (theta**scale)
    out = torch.einsum("...n,d->...nd", pos, omega)
    out = torch.stack([torch.cos(out), -torch.sin(out), torch.sin(out), torch.cos(out)], dim=-1)
    out = rearrange(out, "b n d (i j) -> b n d i j", i=2, j=2)
    return out.float()

def apply_rope(xq: torch.Tensor, xk: torch.Tensor, freqs_cis: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    xq_ = xq.float().reshape(*xq.shape[:-1], -1, 1, 2)
    xk_ = xk.float().reshape(*xk.shape[:-1], -1, 1, 2)
    xq_out = freqs_cis[..., 0] * xq_[..., 0] + freqs_cis[..., 1] * xq_[..., 1]
    xk_out = freqs_cis[..., 0] * xk_[..., 0] + freqs_cis[..., 1] * xk_[..., 1]
    return xq_out.reshape(*xq.shape).type_as(xq), xk_out.reshape(*xk.shape).type_as(xk)

NameError: name 'nn' is not defined

In [1]:
rope(toch.randn((3, 2)), dim=2, theta=2)

NameError: name 'rope' is not defined

In [ ]:
class AdaLN(nn.Module):
    def __init__(self, normalized_shape: int):
        """
        x: (B, N, D)
        scale/shift: (B, 1, D)
        """
        super().__init__()
        self.ln = nn.LayerNorm(normalized_shape=normalized_shape, elementwise_affine=False)
    
    def forward(self, x: torch.Tensor, scale: torch.Tensor, shift: torch.Tensor) -> torch.Tensor:
        x = self.ln(x)
        return x * (1 + scale) + shift

class DoubleStreamAttention(nn.Module): # TODO: RoPE 
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        
        self.n_heads = n_heads
        self.qkv_image_weights = nn.Linear(d_model, 3 * d_model, bias=False)
        self.qkv_text_weights = nn.Linear(d_model, 3 * d_model, bias=False)
        
        self.proj_image = nn.Linear(d_model, d_model)
        self.proj_text = nn.Linear(d_model, d_model)
    
    def norm(self, x: torch.Tensor) -> torch.Tensor:
        return F.rms_norm(x, (x.size(-1),))
    
    def forward(self, data: torch.Tensor, cond: torch.Tensor, pe: torch.Tensor) -> torch.Tensor:
        q_image, k_image, v_image = rearrange(self.qkv_image_weights(data), "b t (k nh hs) -> k b nh t hs", k=3, nh=self.n_heads).unbind(0)
        q_text, k_text, v_text = rearrange(self.qkv_text_weights(cond), "b t (k nh hs) -> k b nh t hs", k=3, nh=self.n_heads).unbind(0)
        
        # b, nh, t, hs dim = 2
        dim = 2
        Q = torch.cat([self.norm(q_image), self.norm(q_text)], dim = dim)
        K = torch.cat([self.norm(k_image), self.norm(k_text)], dim = dim)
        V = torch.cat([v_image, v_text], dim = dim)
        
        Q, K = apply_rope(Q, K, pe)
        attn_outs = rearrange(F.scaled_dot_product_attention(Q, K, V), 'b nh t hs -> b t (nh hs)')
        
        N = data.size(1)
        attn_image, attn_text = attn_outs[:, :N], attn_outs[:, N:]
        
        return self.proj_image(attn_image), self.proj_text(attn_text)

class mmDiTBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        """
        data (B, N, D)
        cond (B, T, D)
        time_steps (B, D)
        """
        super().__init__()
        
        self.adaln_mlp_text = nn.Sequential(nn.SiLU(), nn.Linear(d_model, 6 * d_model))
        self.adaln_mlp_image = nn.Sequential(nn.SiLU(), nn.Linear(d_model, 6 * d_model))
        for mlp in (self.adaln_mlp_text, self.adaln_mlp_image):
            nn.init.zeros_(mlp[1].weight)
            nn.init.zeros_(mlp[1].bias)
        
        self.adaln_text_1 = AdaLN(d_model)
        self.adaln_image_1 = AdaLN(d_model)
        
        self.double_stream_attn = DoubleStreamAttention(d_model, n_heads)
        
        self.adaln_text_2 = AdaLN(d_model)
        self.mlp_text = MLP(d_model, d_model)
        self.adaln_image_2 = AdaLN(d_model)
        self.mlp_image = MLP(d_model, d_model)
        
        
    def forward(self, data: torch.Tensor, cond: torch.Tensor, time_steps: torch.tensor):
        cond_text_scalars = rearrange(self.adaln_mlp_text(time_steps), "b (n d) -> b n d", n=6)
        cond_image_scalars = rearrange(self.adaln_mlp_image(time_steps), "b (n d) -> b n d", n=6)
        
        alpha_text, beta_text, gamma_text, delta_text, epsilon_text, zeta_text = cond_text_scalars.chunk(6, dim=1)
        alpha_image, beta_image, gamma_image, delta_image, epsilon_image, zeta_image = cond_image_scalars.chunk(6, dim=1)
    
        # (QKV)
        attn_image, attn_text = self.double_stream_attn(
            self.adaln_image_1(data, alpha_image, beta_image),
            self.adaln_text_1(cond, alpha_text, beta_text)
        )
        
        cond = gamma_text * attn_text + cond
        data = gamma_image * attn_image + data
        
        cond = zeta_text * self.mlp_text(self.adaln_text_2(cond, delta_text, epsilon_text)) + cond
        data = zeta_image * self.mlp_image(self.adaln_image_2(data, delta_image, epsilon_image)) + data
        
        return data, cond

Main Model

In [ ]:
class DiT(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_blocks: int, patch_size: int, color_channels: int, height: int, width: int, t_channels: int, pooled_dim: int, t5_dim: int):
        """
        inputs:
        data (B, C, H, W)
        t5_cond (B, T, 512)
        
        time_steps (B)
        pooled_cond (B, 1280)
        """
        super().__init__()
        
        self.patchify = Patchify2D(d_model, patch_size, color_channels)
        self.depatchify = Depatchify2D(d_model, patch_size, color_channels, height, width)
        
        self.time_embed = nn.Sequential(
            FourierTimeEmbedding(t_channels),
            MLP(t_channels, d_model)
        )
        
        self.pooled_mlp = MLP(pooled_dim, d_model)
        self.t5_proj = nn.Linear(t5_dim, d_model)
        
        self.pe_embedder = EmbedND(dim=pe_dim, theta=params.theta, axes_dim=params.axes_dim)
        self.mmDiTBlocks = nn.ModuleList([mmDiTBlock(d_model, n_heads) for _ in range(n_blocks)])
        
        self.mlp_adaln = nn.Sequential(nn.SiLU(), nn.Linear(d_model, 2 * d_model))
        self.final_adaln = AdaLN(d_model)
        
    def forward(self, data: torch.Tensor, t5_cond: torch.Tensor, time_steps: torch.Tensor, pooled_cond: torch.Tensor) -> torch.Tensor: # timesteps: torch.Tensor, condition: torch.Tensor)
        data = self.patchify(data)
        t5_cond = self.t5_proj(t5_cond)
        time_steps = self.time_embed(time_steps) + self.pooled_mlp(pooled_cond)
        
        pe = self.pe_embedder(data)
        for block in self.mmDiTBlocks:
            data, t5_cond = block(data, t5_cond, time_steps, pe)
        
        final_scale, final_shift = rearrange(self.mlp_adaln(time_steps), "b (n d) -> b n d", n=2).chunk(2, dim=1)
        data = self.final_adaln(data, final_scale, final_shift)
        return self.depatchify(data)

For testing purposes:

Don't import T5CLIP and load from the the save tensors.

In [4]:
debugging = True

if debugging is False:
    print("importing T5 CLIP encoders")
    from T5CLIP import ConditioningEncoders

    T5CLIP = ConditioningEncoders(device=device)
    print(T5CLIP.pooled_dim, T5CLIP.t5_dim)
    captions = np.load("minidata/smallCaptions.npy").tolist()
    
    t5, pooled = T5CLIP.encode(captions=captions)
else:
    t5clipdata = torch.load("minidata/t5clipsmall.pt", weights_only=True)
    t5 = t5clipdata['t5'].to(device).to(torch.float32)
    pooled = t5clipdata['pooled'].to(device).to(torch.float32)

In [31]:
model = DiT(
    d_model=256,
    n_heads=4,
    n_blocks=1,
    patch_size=16,
    color_channels=3, 
    height=256, 
    width=256, 
    t_channels=32,
    pooled_dim=pooled.shape[-1], # T5CLIP.pooled_dim,
    t5_dim=t5.shape[-1] # T5CLIP.t5_dim
).to(device=device)
print(sum(p.numel() for p in model.parameters() if p.requires_grad), "total trainable params")
# model.load_state_dict(torch.load("10k.pt", weights_only=True))

2505219 total trainable params


In [ ]:
data = torch.load("minidata/smallImageTensors.pt", weights_only=True).to(device) # pass through VAE

In [32]:
with torch.no_grad():
    time_steps = torch.rand(data.size(0), device=device)
    epsilon = torch.randn_like(data, device=device)
    # t * z + (1 - t) * epsilon
    interpolated = torch.einsum('b, b c h w -> b c h w', time_steps, data) + torch.einsum('b, b c h w -> b c h w', 1 - time_steps, epsilon)
    output = model(interpolated, t5, time_steps, pooled)

In [18]:
with torch.no_grad():
    d = model.patchify(data)
    scale, shift = rearrange(model.mlp_adaln(model.time_embed(time_steps)), "b (n d) -> b n d", n=2).chunk(2, dim=1)
    print(d.shape)
    print(model.final_adaln(d, scale, shift).shape)

torch.Size([16, 256, 256])
torch.Size([16, 256, 256])


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for step in range(1000):
    time_steps = torch.rand(data.size(0), device=device)
    epsilon = torch.randn_like(data, device=device)
    # t * z + (1 - t) * epsilon
    interpolated = torch.einsum('b, b c h w -> b c h w', time_steps, data) + torch.einsum('b, b c h w -> b c h w', 1 - time_steps, epsilon)

    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        output = model(interpolated, t5, time_steps, pooled)
    loss = ((output - (data - epsilon))**2).mean()
    
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if step % 100 == 0:
        print(loss.item())
# torch.save(model.state_dict(), "10k.pt")

0.7041312456130981
0.7105070352554321
0.7335554957389832
0.7036372423171997
0.7101303935050964
0.7000738978385925
0.6978013515472412
0.7246826887130737
0.7118179798126221
0.7147535681724548


In [ ]:
from torchvision.transforms import ToTensor, ToPILImage, transforms
import matplotlib.pyplot as plt
from ipywidgets import interact, Layout, IntSlider
# from IPython.display import display
to_tensor = ToTensor()
to_image = ToPILImage()

def display(tensor):
    img = rearrange(tensor, "1 c h w -> h w c",).cpu()
    plt.figure(figsize=(8, 8))
    plt.imshow(img)

steps = 100
@torch.no_grad()
def sample_image():
    outputs = []
    X_0 = epsilon # torch.randn((1, 3, 32, 32), device=device)
    
    # null_classification = create_classification(10)
    # t_t5, t_pooled = T5CLIP.encode(captions=caption)
    for _, t in enumerate(torch.linspace(0, 1, steps, device=device), start=1):
        outputs.append(X_0)
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            X_0 = X_0 + (1 / steps) * model(X_0, t5.expand(16, -1, -1), t.view(1), pooled.expand(16, -1))
    return outputs # X_0, ..., X_1


outputs = sample_image()
def display_image(step, idx):
    display(outputs[step][idx].unsqueeze(0))

interact(display_image, step=IntSlider(min=0, max=steps-1), idx=IntSlider(min=0, max=15), layout=Layout(width='75%'))

interactive(children=(IntSlider(value=0, description='step', max=99), IntSlider(value=0, description='idx', ma…

<function __main__.display_image(step, idx)>

### random interpolation and image displaying

In [ ]:
max_steps = 100
@torch.no_grad()
def show_image_interpolation(t, batch_idx):
    t = torch.tensor(t / max_steps, device=device)
    X_i = t.view(-1, 1, 1, 1) * data + (1 - t.view(-1, 1, 1, 1)) * epsilon
    X_i = rearrange(X_i[batch_idx], "c h w -> h w c").cpu()
    
    print(captions[batch_idx])
    plt.figure(figsize=(8, 8))
    plt.imshow(X_i) #, vmin=0, vmax=1)
    
interact(show_image_interpolation, t=IntSlider(min=0, max=(max_steps)), batch_idx=IntSlider(min=0, max=15), layout=Layout(width='75%'))

interactive(children=(IntSlider(value=0, description='t'), IntSlider(value=0, description='batch_idx', max=15)…

<function __main__.show_image_interpolation(t, batch_idx)>